# Tutorial 6: Text Embeddings

This notebook walks through the `embpy` text-embedding stack end-to-end.
Text embeddings let you encode gene descriptions, drug mechanism-of-action
notes, paper abstracts, cell-line metadata, or any free-text annotation
into the same vector space as your sequence and image embeddings. They
are the cheap "metadata channel" of a multi-modal embedding pipeline.

The notebook is organised so that every section answers a single
question:

| Section | Question |
|---|---|
| 0 | What models, panels, and seeds are we using? |
| 1 | Which text models does `embpy` expose and how do their pooling strategies differ? |
| 2 | How do `embed_text`, `embed_texts_batch`, and the API path behave? |
| 3 | What does each knowledge source (`MyGene`, `NCBI`, `Ensembl`, `UniProt`, `Wikipedia`, `PubChem`) actually contribute? |
| 4 | Do gene-description embeddings separate genes from cytokines on UMAP? |
| 5 | Do drug-description embeddings cluster by mechanism-of-action? |
| 6 | Do the UMAP groupings agree with curated ChEMBL targets / mechanism-of-action? |
| 7 | How do different text models compare on the same input? |
| 8 | Which knowledge source dominates the combined embedding? |
| 9 | Summary table of every function the notebook used. |

Runs end-to-end on CPU in roughly three minutes on a warm cache. No
GPU, no `HF_TOKEN`, and no `OPENAI_API_KEY` required; cells that would
need them fall back to "skipped" messages.

## Preferred workflow: text embeddings as contextual entity features

Text encoders are most useful when their outputs remain traceable to the original biological entity and source text. The standardized workflow gives that traceability:

1. Use `BioEmbedder.embed(..., entity_type="text")` for direct descriptions, abstracts, mechanism notes, or structured prompts.
2. Canonical text/input IDs become the primary keys; original text labels and source metadata stay as aliases/provenance.
3. Put reusable description embeddings into `EmbeddingStore`/`.emstore`, then use `adata.embpy` to compare text geometry against gene, protein, molecule, morphology, or phenotype embeddings.

This notebook still shows the lower-level text methods because they are useful for source ablations and prompt engineering. The registry/accessor layer is what makes those embeddings auditable and comparable later.


In [ ]:
from embpy import BioEmbedder
from embpy.store import EmbeddingStore

RUN_STANDARDIZED_EMBED_DEMO = False

if RUN_STANDARDIZED_EMBED_DEMO:
    embedder = BioEmbedder(device="auto")

    text_adata = embedder.embed(
        [
            "TP53 is a tumor suppressor involved in DNA damage response.",
            "Imatinib inhibits BCR-ABL and related tyrosine kinases.",
        ],
        entity_type="text",
        model="minilm_l6_v2",
        output="anndata",
    )
    print("text embeddings in .obsm:", list(text_adata.obsm.keys()))

# For reusable text/context embeddings:
# store = EmbeddingStore.from_results(text_result)
# adata.embpy.register_store(store, name="entity_context")
# context = adata.embpy.prompt_context(output="markdown")


## 0. Setup

Imports, deterministic seeds, a single shared `BioEmbedder`, and the small
entity panels reused across every section. Panels are deliberately small
(5-10 items each) so each section finishes in seconds, not minutes; the
runtime budget for the whole notebook is well under three minutes on CPU.

In [1]:
import logging
import os
import random
import time
import warnings
from typing import Any

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.WARNING)
warnings.filterwarnings("ignore")
sns.set_theme(context="notebook", style="whitegrid", palette="muted")
%matplotlib inline

# Seeds: numpy + python random for any non-deterministic helper; UMAP
# uses random_state=0 explicitly downstream.
random.seed(0)
np.random.seed(0)
SEED = 0

# Optional Layer-4 timing helper. Falls back to time.perf_counter when
# embpy.observability is not importable, so the notebook still runs.
try:
    from embpy.observability import time_block

    HAS_TIME_BLOCK = True
except Exception:
    HAS_TIME_BLOCK = False
    from contextlib import contextmanager

    @contextmanager
    def time_block(event, **fields):
        t0 = time.perf_counter()
        ctx = dict(fields)
        try:
            yield ctx
        finally:
            ctx["latency_ms"] = (time.perf_counter() - t0) * 1000.0


print(f"numpy={np.__version__}  pandas={pd.__version__}  seed={SEED}  time_block={HAS_TIME_BLOCK}")

numpy=2.4.3  pandas=2.3.3  seed=0  time_block=True


In [ ]:
from embpy.embedder import BioEmbedder

embedder = BioEmbedder(device="auto")
print(f"Device: {embedder.device}")

# Reusable entity panels. Kept small on purpose so every cell finishes
# in seconds; bigger panels belong in a SLURM job, not in a tutorial.
GENES = ["TP53", "BRCA1", "EGFR", "MYC", "KRAS", "STAT1", "IRF1"]
CYTOKINES = ["IL6", "IL1B", "TNF", "IFNG", "IL10", "TGFB1", "IL2"]
DRUGS_BY_MOA = {
    "EGFR_inhibitor": ["gefitinib", "erlotinib", "osimertinib"],
    "BRAF_inhibitor": ["vemurafenib", "dabrafenib", "encorafenib"],
    "HDAC_inhibitor": ["vorinostat", "panobinostat", "romidepsin"],
    "mTOR_inhibitor": ["rapamycin", "everolimus", "temsirolimus"],
    "COX_inhibitor": ["aspirin", "ibuprofen", "naproxen"],
}
CELLLINES = ["A549", "HeLa", "MCF7", "HCT116", "K562"]

print(
    f"Panels -> genes:{len(GENES)}  cytokines:{len(CYTOKINES)}  "
    f"drugs:{sum(len(v) for v in DRUGS_BY_MOA.values())} "
    f"(across {len(DRUGS_BY_MOA)} MoAs)  celllines:{len(CELLLINES)}"
)

# Description cache. Keyed by (entity_type, identifier, sources_tuple)
# so re-running cells is idempotent and we never hammer MyGene / NCBI
# more than once per (entity, source-set) combo per notebook session.
DESC_CACHE: dict[tuple, Any] = {}


def cached_get_description(entity_type: str, identifier: str, sources: tuple[str, ...] = ()) -> dict[str, str]:
    """Wrapper around TextResolver.get_description with notebook-local cache.

    Returns the raw per-source dict (source -> text) for inspection.
    Missing sources surface as empty strings, not exceptions.
    """
    key = (entity_type, identifier, sources)
    if key in DESC_CACHE:
        return DESC_CACHE[key]
    try:
        descs = embedder.text_resolver.get_description(
            identifier,
            entity_type=entity_type,
            sources=list(sources) if sources else "all",
        )
    except Exception as e:
        # Per-identifier failure must not nuke the whole cell.
        logging.warning("get_description failed for %r (%s): %s", identifier, entity_type, e)
        descs = {}
    DESC_CACHE[key] = descs
    return descs


print(f"DESC_CACHE ready (currently {len(DESC_CACHE)} entries).")

## 1. The text-model surface in embpy

`embpy` registers two families of text encoders:

* **Sentence-encoder / BERT family** wrapped by `TextLLMWrapper`
  (`minilm_l6_v2`, `bert_base_uncased`). These use bidirectional
  attention; you typically pool with `mean` (sentence-transformers
  convention) or `cls` (canonical BERT readout).
* **LLaMA-style decoder family** wrapped by `LlamaEmbeddingWrapper`
  (`llama3.2_1b`, `llama3.2_3b`, `llama3.1_8b`). Causal models cannot
  attend bidirectionally so the `cls` token has no semantic meaning;
  the right readout is `last_token` (the final hidden state after the
  full prompt is consumed). Gated models -- a HuggingFace token in
  `HF_TOKEN` is required to download them.

Pooling cheat sheet:

| Strategy | When to use |
|---|---|
| `mean`       | Default for BERT / sentence-transformers; robust to length. |
| `cls`        | BERT-family with a trained CLS readout (classification setups). |
| `last_token` | Decoder-only LLMs (LLaMA, GPT-style). |
| `max`        | Rare; emphasises peak features, sensitive to outlier tokens. |
| `none`       | Return the full token matrix; downstream code does its own pooling. |

In [ ]:
from embpy.embedder_registry.text import TEXT_MODELS
from embpy.models.text_models import LlamaEmbeddingWrapper, TextLLMWrapper

wrapper_name = {TextLLMWrapper: "TextLLMWrapper", LlamaEmbeddingWrapper: "LlamaEmbeddingWrapper"}

rows = []
for key, (cls, hf_path) in TEXT_MODELS.items():
    rows.append(
        {
            "registry_key": key,
            "wrapper": wrapper_name.get(cls, cls.__name__ if cls else "<none>"),
            "hf_path": hf_path,
        }
    )
text_models_df = pd.DataFrame(rows)
print(text_models_df.to_string(index=False))

print()
print(f"TextLLMWrapper pooling strategies:        {TextLLMWrapper.available_pooling_strategies}")
print(f"LlamaEmbeddingWrapper pooling strategies: {LlamaEmbeddingWrapper.available_pooling_strategies}")

In [ ]:
# Same sentence, four pooling strategies, single model. The point is to
# convince yourself the strategies are NOT interchangeable: vectors differ
# in norm and direction even for the same input.
SENTENCE = "TP53 encodes the tumor suppressor protein p53. It responds to cellular stress and regulates the cell cycle."
POOLINGS = ["mean", "max", "cls"]

pooling_rows = []
for strat in POOLINGS:
    try:
        vec = embedder.embed_text(SENTENCE, model="minilm_l6_v2", pooling_strategy=strat)
        pooling_rows.append(
            {
                "pooling": strat,
                "dim": int(vec.shape[0]),
                "l2_norm": float(np.linalg.norm(vec)),
                "dtype": str(vec.dtype),
            }
        )
    except Exception as e:
        # MiniLM has no trained CLS readout; print the failure
        # instead of swallowing -- the point is to show WHAT differs.
        logging.warning("pooling %r failed: %s", strat, e)
        pooling_rows.append({"pooling": strat, "dim": None, "l2_norm": None, "dtype": "error"})

pd.DataFrame(pooling_rows)

## 2. Single-text and batched embeddings

Three public entry points on `BioEmbedder`:

* `embed_text(text, model)` -- one string in, one vector out.
* `embed_texts_batch(texts, model, batch_size)` -- length-N list in,
  length-N list out. Inputs that fail come back as `None` (never
  silently dropped).
* `embed_text_api(text, provider="openai", ...)` -- talks to a hosted
  embedding endpoint instead of running a local model.

The API path is guarded on `OPENAI_API_KEY`; without the env var the
cell prints "skipped" and the notebook continues.

In [ ]:
# Single text -> single vector.
vec = embedder.embed_text(SENTENCE, model="minilm_l6_v2", pooling_strategy="mean")
print(f"embed_text -> shape={tuple(vec.shape)} dtype={vec.dtype} l2={float(np.linalg.norm(vec)):.3f}")

# 10 short biology sentences, batched. Use a chunk size below the
# input count to actually exercise the chunking path inside the
# wrapper.
SENTENCES = [
    "TP53 regulates the cell cycle and apoptosis.",
    "BRCA1 is involved in homologous recombination DNA repair.",
    "EGFR signaling promotes proliferation in many epithelial tumors.",
    "MYC is a master transcriptional regulator of growth.",
    "KRAS is a small GTPase frequently mutated in pancreatic cancer.",
    "IL6 is a pleiotropic pro-inflammatory cytokine.",
    "Interferon gamma activates macrophages and induces MHC class II.",
    "Rapamycin inhibits mTORC1 and extends lifespan in model organisms.",
    "Aspirin irreversibly acetylates cyclooxygenase enzymes.",
    "Imatinib is a BCR-ABL tyrosine kinase inhibitor.",
]

with time_block("embed_texts_batch", model="minilm_l6_v2", n_inputs=len(SENTENCES), batch_size=4) as ev:
    batch = embedder.embed_texts_batch(
        SENTENCES,
        model="minilm_l6_v2",
        pooling_strategy="mean",
        batch_size=4,
    )
n_ok = sum(1 for x in batch if x is not None)
dims = {tuple(x.shape) for x in batch if x is not None}
print(
    f"embed_texts_batch -> {n_ok}/{len(SENTENCES)} ok, "
    f"dims={dims}, latency_ms={ev.get('latency_ms', 0):.1f}, "
    f"throughput={n_ok / max(ev.get('latency_ms', 1e-6) / 1000, 1e-6):.1f} texts/s"
)

In [ ]:
# API path: skipped cleanly without OPENAI_API_KEY.
if os.environ.get("OPENAI_API_KEY"):
    try:
        api_vec = embedder.embed_text_api(
            SENTENCE,
            model="text-embedding-3-small",
            provider="openai",
        )
        print(
            f"embed_text_api(openai) -> shape={tuple(api_vec.shape)} "
            f"dtype={api_vec.dtype} l2={float(np.linalg.norm(api_vec)):.3f}"
        )
    except Exception as e:
        # Per-call failure is logged but does not crash the notebook.
        logging.warning("embed_text_api failed: %s", e)
        print(f"embed_text_api(openai) -> error: {e}")
else:
    print("embed_text_api(openai) -> skipped, no OPENAI_API_KEY in env")

## 3. Description sources via `TextResolver`

`embpy.resources.text.TextResolver` is the layer between a biological
identifier (a gene symbol, a UniProt accession, a drug name) and the
free text the embedding model actually sees. It queries multiple public
APIs in parallel and returns a `dict` of `{source: text}`:

* genes -- MyGene, NCBI, Ensembl, Wikipedia
* proteins -- UniProt, Wikipedia
* molecules -- PubChem, Wikipedia
* cell lines -- Cellosaurus / DepMap / Cell Model Passports
  (via `CellLineAnnotator`)

`BioEmbedder.embed_description` and `embed_descriptions_batch` are the
thin convenience wrappers that combine the per-source dict into a
single string via `get_combined_description`, then forward it to the
chosen text model.

The four cells below walk through (3a) per-source content, (3b) drug
and protein variants, (3c) the combined string that actually gets
embedded, and (3d) per-source cosine similarity for the same gene
(sanity check: independent sources should agree).

In [ ]:
# 3a. Per-source descriptions for ONE gene. Source coverage table
# shows which APIs returned content and how much.
gene_sources = cached_get_description("gene", "TP53")
print(f"Gene 'TP53' source coverage ({len(gene_sources)} sources queried):")
for src, txt in gene_sources.items():
    ok = "OK" if txt else "EMPTY"
    print(f"  {src:10s} [{ok:5s}] len={len(txt):5d}  preview: {txt[:200].strip()!r}")

In [ ]:
# 3b. Same idea for a drug (PubChem + Wikipedia) and a protein
# (UniProt + Wikipedia). Notice that the same identifier can be
# routed through different source sets depending on entity_type.
drug_sources = cached_get_description("molecule", "imatinib")
print("Drug 'imatinib' source coverage:")
for src, txt in drug_sources.items():
    ok = "OK" if txt else "EMPTY"
    print(f"  {src:10s} [{ok:5s}] len={len(txt):5d}  preview: {txt[:160].strip()!r}")

print()
protein_sources = cached_get_description("protein", "EGFR")
print("Protein 'EGFR' source coverage:")
for src, txt in protein_sources.items():
    ok = "OK" if txt else "EMPTY"
    print(f"  {src:10s} [{ok:5s}] len={len(txt):5d}  preview: {txt[:160].strip()!r}")

In [ ]:
# 3c. Combined description == the actual input to the text model.
# This is what embed_description will hand to MiniLM / BERT / LLaMA.
combined = embedder.text_resolver.get_combined_description(
    "TP53",
    entity_type="gene",
    sources="all",
)
print(f"Combined description length: {len(combined)} chars")
print("Preview (first 600 chars):")
print(combined[:600], "..." if len(combined) > 600 else "")

# Sanity: embed_description should pass the combined string through
# the model and return a (384,)-dim MiniLM vector. We just check the
# shape and norm; the actual vector content is tested in 3d.
vec_combined = embedder.embed_description("TP53", model="minilm_l6_v2", entity_type="gene")
print(f"\nembed_description('TP53') -> shape={tuple(vec_combined.shape)} l2={float(np.linalg.norm(vec_combined)):.3f}")

In [ ]:
# 3d. Embed each non-empty source SEPARATELY for TP53 and compute the
# pairwise cosine similarity matrix. Independent knowledge sources
# describing the same gene should agree -> off-diagonal values close
# to 1.0. If a source is way off, that source is contributing
# something semantically different (worth investigating before
# trusting downstream UMAPs).
from sklearn.metrics.pairwise import cosine_similarity

per_source_vecs: dict[str, np.ndarray] = {}
for src, txt in gene_sources.items():
    if not txt:
        continue
    try:
        per_source_vecs[src] = embedder.embed_text(
            txt,
            model="minilm_l6_v2",
            pooling_strategy="mean",
        )
    except Exception as e:
        # Don't let one source kill the cell.
        logging.warning("embed_text failed for source %r of TP53: %s", src, e)

src_names = list(per_source_vecs.keys())
mat = np.stack([per_source_vecs[s] for s in src_names], axis=0)
sim = cosine_similarity(mat)
print(f"Per-source cosine similarity (n={len(src_names)} sources):")
print(pd.DataFrame(sim, index=src_names, columns=src_names).round(3))

fig, ax = plt.subplots(figsize=(4.5, 3.5))
sns.heatmap(
    sim,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    vmin=0,
    vmax=1,
    xticklabels=src_names,
    yticklabels=src_names,
    ax=ax,
    cbar_kws={"label": "cosine"},
)
ax.set_title("TP53: per-source agreement\n(off-diagonal -> 1.0 means sources agree)")
plt.tight_layout()
plt.show()

## 4. UMAP of gene-description embeddings vs cytokine-description embeddings

Sanity check that the resolver + embedder pipeline carries biological
signal: cytokines (`IL6`, `TNF`, `IFNG`, ...) describe the immune
response and should land near each other in MiniLM space, separated
from tumor-suppressor / oncogene / kinase entries. We build a small
`AnnData`, compute UMAP via `embpy.tl.compute_umap`, and plot the
embedding with the panel label as colour.

In [ ]:
import anndata as ad
from embpy.tl import compute_umap

gene_panel = list(GENES) + list(CYTOKINES)
panel_label = ["gene"] * len(GENES) + ["cytokine"] * len(CYTOKINES)

with time_block("embed_descriptions_batch", model="minilm_l6_v2", n_inputs=len(gene_panel), entity_type="gene") as ev:
    embs = embedder.embed_descriptions_batch(
        gene_panel,
        model="minilm_l6_v2",
        entity_type="gene",
        sources="all",
        batch_size=8,
    )
print(
    f"embed_descriptions_batch -> {sum(e is not None for e in embs)}/{len(embs)} ok in {ev.get('latency_ms', 0):.0f} ms"
)

# Drop None rows; log which inputs were dropped so the user can audit.
keep_names: list[str] = []
keep_labels: list[str] = []
keep_excerpts: list[str] = []
keep_vecs: list[np.ndarray] = []
for name, label, vec in zip(gene_panel, panel_label, embs, strict=False):
    if vec is None:
        logging.warning("Dropping %r (%s): no embedding returned", name, label)
        continue
    keep_names.append(name)
    keep_labels.append(label)
    descs = cached_get_description("gene", name)
    # Excerpt for inspection in adata.obs; first non-empty source.
    excerpt = next((t[:160] for t in descs.values() if t), "")
    keep_excerpts.append(excerpt)
    keep_vecs.append(np.asarray(vec, dtype=np.float32).reshape(-1))

X = np.stack(keep_vecs, axis=0)
adata_genes = ad.AnnData(
    X=np.zeros((len(keep_names), 1), dtype=np.float32),  # placeholder .X
    obs=pd.DataFrame(
        {
            "name": keep_names,
            "panel": pd.Categorical(keep_labels, categories=["gene", "cytokine"]),
            "description_excerpt": keep_excerpts,
        },
        index=keep_names,
    ),
)
adata_genes.obsm["X_text"] = X
print(
    f"adata_genes -> {adata_genes.n_obs} obs, "
    f"X_text shape={adata_genes.obsm['X_text'].shape}, "
    f"panels={dict(adata_genes.obs['panel'].value_counts())}"
)

In [ ]:
import scanpy as sc

# n_neighbors must be < n_obs; pick a small value safe for tiny panels.
n_neighbors = min(8, max(2, adata_genes.n_obs - 1))
compute_umap(adata_genes, obsm_key="X_text", n_neighbors=n_neighbors, output_key="X_umap_text")
# compute_umap also writes the scanpy-standard 'X_umap' key, which is
# what sc.pl.umap reads by convention.
print("UMAP keys in obsm:", [k for k in adata_genes.obsm if "umap" in k.lower()])

with plt.rc_context({"figure.figsize": (5, 4)}):
    sc.pl.umap(adata_genes, color="panel", title="Text embeddings: genes vs cytokines", show=True, frameon=False)

## 5. UMAP of drug descriptions coloured by mechanism of action

If text descriptions encode pharmacology, drugs targeting the same
protein should land near each other. We embed the five-MoA panel and
plot two views: (left) coloured by MoA, (right) coloured by the L2
norm of the embedding -- a quick health check that the description
length is not driving the picture. Then we quantify with mean cosine
similarity within-MoA vs between-MoA. Within > between is the success
metric.

In [ ]:
drug_names: list[str] = []
drug_moa: list[str] = []
for moa, ds in DRUGS_BY_MOA.items():
    drug_names.extend(ds)
    drug_moa.extend([moa] * len(ds))

with time_block("embed_descriptions_batch", entity_type="molecule", n_inputs=len(drug_names)) as ev:
    drug_embs = embedder.embed_descriptions_batch(
        drug_names,
        model="minilm_l6_v2",
        entity_type="molecule",
        sources="all",
        batch_size=8,
    )
print(
    f"embed_descriptions_batch (drugs) -> "
    f"{sum(e is not None for e in drug_embs)}/{len(drug_embs)} ok "
    f"in {ev.get('latency_ms', 0):.0f} ms"
)

# Drop unresolved; log which.
keep_n: list[str] = []
keep_m: list[str] = []
keep_v: list[np.ndarray] = []
for name, moa, vec in zip(drug_names, drug_moa, drug_embs, strict=False):
    if vec is None:
        logging.warning("Dropping drug %r (MoA=%s): no embedding", name, moa)
        continue
    keep_n.append(name)
    keep_m.append(moa)
    keep_v.append(np.asarray(vec, dtype=np.float32).reshape(-1))

Xd = np.stack(keep_v, axis=0)
adata_drugs = ad.AnnData(
    X=np.zeros((len(keep_n), 1), dtype=np.float32),
    obs=pd.DataFrame(
        {
            "name": keep_n,
            "moa": pd.Categorical(keep_m),
            "l2_norm": np.linalg.norm(Xd, axis=1),
        },
        index=keep_n,
    ),
)
adata_drugs.obsm["X_text"] = Xd
print(f"adata_drugs -> {adata_drugs.n_obs} obs, MoA distribution: {dict(adata_drugs.obs['moa'].value_counts())}")

In [ ]:
n_neighbors_drugs = min(6, max(2, adata_drugs.n_obs - 1))
compute_umap(adata_drugs, obsm_key="X_text", n_neighbors=n_neighbors_drugs, output_key="X_umap_text")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sc.pl.umap(
    adata_drugs, color="moa", ax=axes[0], show=False, title="Drugs coloured by mechanism of action", frameon=False
)
sc.pl.umap(
    adata_drugs,
    color="l2_norm",
    ax=axes[1],
    show=False,
    title="L2 norm of embedding (sanity)",
    frameon=False,
    color_map="viridis",
)
plt.tight_layout()
plt.show()

# Quantify clustering quality with within- vs between-MoA cosine sim.
sim_d = cosine_similarity(Xd)
n = len(keep_m)
moa_arr = np.asarray(keep_m)
same = moa_arr[:, None] == moa_arr[None, :]
diag = np.eye(n, dtype=bool)
within = sim_d[same & ~diag]
between = sim_d[~same]
print(f"Within-MoA  cosine sim: mean={within.mean():.3f}  n_pairs={within.size}")
print(f"Between-MoA cosine sim: mean={between.mean():.3f}  n_pairs={between.size}")
print(
    f"Within - Between gap: {within.mean() - between.mean():+.3f}  (positive => text descriptions carry MoA structure)"
)

## 6. Cross-check text embeddings against structured ChEMBL annotations

The UMAP in section 5 looks like drugs group by MoA, but UMAPs can lie.
A stronger check is to ask: do the text descriptions agree with
**curated** annotations independent of any embedding? `MoleculeAnnotator`
pulls mechanism-of-action records and target proteins straight from
ChEMBL, which is the ground-truth source for "what does this drug
hit?". If the text-derived UMAP and ChEMBL agree, text descriptions
are a usable cheap proxy for MoA.

Liu et al., "Learning Perturbation Effects Through Contrastive
Alignment of Transcriptomics and Textual Embeddings", MLGenX 2026
(https://openreview.net/forum?id=WcWfmx0ILj) motivates using curated
text descriptions of genes and compounds as a complementary modality
to transcriptomics; this notebook produces exactly that text side.

In [ ]:
from embpy.resources import DrugResolver
from embpy.resources.molecule.annotator import MoleculeAnnotator

drug_resolver = DrugResolver()
annotator = MoleculeAnnotator()

cross_rows: list[dict[str, Any]] = []
for name, moa in zip(keep_n, keep_m, strict=False):
    # Resolve name -> SMILES. Use DrugResolver first (cheap), fall back
    # to the annotator's private resolver if needed.
    smiles: str | None = None
    try:
        smiles = drug_resolver.name_to_smiles(name)
    except Exception as e:
        logging.warning("DrugResolver failed for %r: %s", name, e)
    if not smiles:
        try:
            smiles = annotator._resolve_to_smiles(name)
        except Exception as e:
            logging.warning("Annotator fallback failed for %r: %s", name, e)
    if not smiles:
        cross_rows.append(
            {"drug": name, "expected_MoA": moa, "smiles": None, "chembl_moa": None, "chembl_targets": None}
        )
        continue

    # Fetch MoA and target list. Each call can fail independently;
    # log and continue so one HTTP miss does not poison the table.
    try:
        moa_records = annotator.get_mechanism_of_action(smiles)
    except Exception as e:
        logging.warning("get_mechanism_of_action(%r) failed: %s", name, e)
        moa_records = []
    try:
        targets = annotator.get_target_proteins(smiles)
    except Exception as e:
        logging.warning("get_target_proteins(%r) failed: %s", name, e)
        targets = []

    cross_rows.append(
        {
            "drug": name,
            "expected_MoA": moa,
            "smiles": smiles[:40] + ("..." if smiles and len(smiles) > 40 else ""),
            "chembl_moa": (moa_records[0].get("mechanism", "") if moa_records else ""),
            "chembl_targets": ", ".join((t.get("target_name") or "?") for t in targets[:3]),
        }
    )

cross_df = pd.DataFrame(cross_rows)
print(cross_df.to_string(index=False, max_colwidth=60))

## 7. Model comparison

The same description goes through three different encoders. We are NOT
saying "the bigger encoder is better" -- that question needs a
downstream task. We just want to (a) show the dimensions differ, (b)
confirm all three produce valid vectors on the same input, and (c)
flag that LLaMA needs `HF_TOKEN` and the right pooling (`last_token`).
The right diagnostic for "is this encoder better for my task?" is
cosine similarity between paraphrases, NOT raw dimension count.

In [ ]:
probe_entities = [
    ("gene", "TP53"),
    ("gene", "IL6"),  # cytokine
    ("molecule", "imatinib"),
]
# Encoder roster: 2 always-on, 1 gated on HF_TOKEN.
ENCODERS: list[tuple[str, str]] = [
    ("minilm_l6_v2", "mean"),
    ("bert_base_uncased", "cls"),
]
if os.environ.get("HF_TOKEN"):
    ENCODERS.append(("llama3.2_1b", "last_token"))
else:
    print("LLaMA path skipped, set HF_TOKEN to enable.")

cmp_rows = []
for entity_type, ident in probe_entities:
    for model_name, pool in ENCODERS:
        try:
            v = embedder.embed_description(
                ident,
                model=model_name,
                entity_type=entity_type,
                pooling_strategy=pool,
            )
            cmp_rows.append(
                {
                    "entity": ident,
                    "type": entity_type,
                    "model": model_name,
                    "pool": pool,
                    "dim": int(v.shape[0]),
                    "l2": float(np.linalg.norm(v)),
                }
            )
        except Exception as e:
            # Failure on one (model, entity) cell must not skip others.
            logging.warning("model=%s entity=%s failed: %s", model_name, ident, e)
            cmp_rows.append(
                {
                    "entity": ident,
                    "type": entity_type,
                    "model": model_name,
                    "pool": pool,
                    "dim": None,
                    "l2": None,
                }
            )

pd.DataFrame(cmp_rows)

## 8. Source-ablation study (interpretability)

Section 3d asked "do sources agree on the same gene?". This section
asks the dual: **which source dominates the combined embedding the
downstream UMAP actually used?** We embed TP53 four times, each time
with a single source enabled, plus once with all four, and report the
cosine similarity of each single-source vector against the "all"
vector. The tallest bar is the source carrying most of the semantic
signal; sources with very low bars are decorative -- removing them
would not change the embedding much.

In [ ]:
SINGLE_SOURCES = ["mygene", "ncbi", "ensembl", "wikipedia"]
ablation_vecs: dict[str, np.ndarray] = {}

# Single-source ablations.
for src in SINGLE_SOURCES:
    try:
        v = embedder.embed_description(
            "TP53",
            model="minilm_l6_v2",
            entity_type="gene",
            sources=[src],
        )
        ablation_vecs[src] = np.asarray(v, dtype=np.float32).reshape(-1)
    except Exception as e:
        logging.warning("Source ablation %r failed: %s", src, e)

# Reference: all sources combined.
try:
    ablation_vecs["ALL"] = np.asarray(
        embedder.embed_description("TP53", model="minilm_l6_v2", entity_type="gene", sources="all"),
        dtype=np.float32,
    ).reshape(-1)
except Exception as e:
    logging.warning("ALL-source embedding failed: %s", e)

ref = ablation_vecs.get("ALL")
if ref is None:
    print("Cannot run ablation: ALL-source embedding missing.")
else:
    # Cosine similarity of each single-source vector against ALL.
    sims = []
    for src in SINGLE_SOURCES:
        if src not in ablation_vecs:
            continue
        sims.append(
            {"source": src, "cos_vs_ALL": float(cosine_similarity(ablation_vecs[src][None, :], ref[None, :])[0, 0])}
        )
    abl_df = pd.DataFrame(sims).sort_values("cos_vs_ALL", ascending=False)
    print(abl_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(5, 3))
    sns.barplot(data=abl_df, x="source", y="cos_vs_ALL", ax=ax, color=sns.color_palette()[0])
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("cos(single source, ALL)")
    ax.set_title("TP53: which source dominates the combined embedding?")
    for i, v in enumerate(abl_df["cos_vs_ALL"]):
        ax.text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)
    plt.tight_layout()
    plt.show()

## 9. Summary

Every public function this notebook touched, in the order it was first used:

| Function | What it does | Section |
|---|---|---|
| `BioEmbedder.embed_text` | Embed one arbitrary string with a text model. | 1, 7 |
| `BioEmbedder.embed_texts_batch` | Batched version, returns `None` per failed input. | 2 |
| `BioEmbedder.embed_text_api` | API-based encoder (OpenAI, Cohere, ...). Skipped without key. | 2 |
| `BioEmbedder.embed_description` | Fetch description for an identifier + embed it. | 3, 7, 8 |
| `BioEmbedder.embed_descriptions_batch` | Batched description -> embedding. | 4, 5 |
| `BioEmbedder.embed_gene` (text path) | Same pipeline but via the gene path with `gene_description_format`. | (referenced, not invoked) |
| `TextResolver.get_gene_description` | Per-source gene text dict. | 3a |
| `TextResolver.get_protein_description` | Per-source protein text dict. | 3b |
| `TextResolver.get_molecule_description` | Per-source molecule text dict. | 3b |
| `TextResolver.get_cellline_description` | Cellosaurus / DepMap text. | (panel exists) |
| `TextResolver.get_combined_description` | Concatenates all sources into the string the model embeds. | 3c |
| `MoleculeAnnotator.get_mechanism_of_action` | Curated ChEMBL MoA records. | 6 |
| `MoleculeAnnotator.get_target_proteins` | Curated ChEMBL targets. | 6 |
| `DrugResolver.name_to_smiles` | Resolve drug name -> SMILES. | 6 |
| `embpy.tl.compute_umap` | UMAP coordinates in `obsm`. | 4, 5 |

**Practical takeaways**

* Always pick a pooling strategy that matches the model family
  (`mean`/`cls` for BERT-style, `last_token` for causal LMs).
* Inspect per-source coverage before trusting a combined-description
  embedding. The section 8 ablation tells you which knowledge source
  is doing the work.
* Use the per-MoA cosine gap (section 5) as a cheap diagnostic: if
  within < between, your text descriptions or model are not capturing
  the structure you care about.

Next: [Tutorial 7 -- PPI Network Embeddings](07_ppi_embeddings.ipynb)